In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Модель для создания векторных представлений
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5") # SentenceTransformer это библиотека для создания векторных представлений предложений и текстов на основе предобученных моделей трансформеров. В данном случае используется модель "BAAI/bge-small-en-v1.5", которая предназначена для генерации эмбеддингов на английском языке, имеет размерность 768

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1364.28it/s]



Используй контекст ниже для ответа на вопрос.

Контекст: График работы компании: с 9:00 до 18:00 по будням.
Вопрос: Как мне подключить ноутбук к интернету в офисе?



In [ ]:
# 2. База знаний нужен для поиска информации по запросу пользователя. В данном случае база знаний состоит из нескольких предложений, содержащих информацию о компании и ее работе.
documents = [
    "График работы компании: с 9:00 до 18:00 по будням.",
    # "Для оформления отпуска нужно подать заявление за 2 недели.",
    "Пароль от интернета в офисе: Corporate_Pass_2026."
]

# Создаем векторы для документов. Весь текст из базы знаний преобразуется в векторное представление, которое затем используется для поиска наиболее релевантного документа по запросу пользователя.
doc_embeddings = embedder.encode(documents)

# 3. Пользовательский запрос
user_query = "Как мне подключить ноутбук к интернету в офисе?"
query_embedding = embedder.encode([user_query]) # Используем тот же эмбеддер 

# 4. Поиск ближайшего по смыслу чанка (Косинусное сходство)
similarities = np.dot(doc_embeddings, query_embedding.T).squeeze()
best_doc_index = np.argmax(similarities)
retrieved_context = documents[best_doc_index]

# 5. Формирование итогового промпта для генеративной модели
final_prompt = f"""
Используй контекст ниже для ответа на вопрос.

Контекст: {retrieved_context}
Вопрос: {user_query}
"""

print(final_prompt)
# Этот промпт подается в Qwen / Llama / FastAPI сервис


Используй контекст ниже для ответа на вопрос.

Контекст: Пароль от интернета в офисе: Corporate_Pass_2026.
Вопрос: Как мне подключить ноутбук к интернету в офисе?



In [ ]:
from qdrant_client import QdrantClient # Qdrant это векторная база данных, которая позволяет хранить и искать векторы (эмбеддинги) с высокой производительностью. Она используется для поиска схожих документов на основе их векторных представлений.
from qdrant_client.models import VectorParams, Distance, PointStruct
from sentence_transformers import SentenceTransformer

# 1. Загружаем модель для создания векторов (384 измерения)
print("Загрузка модели эмбеддингов...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Инициализируем Qdrant в режиме in-memory (без поднятия внешнего сервера).
client = QdrantClient(":memory:")

# 3. Создаем коллекцию под названием "documents". Название позволяет идентифицировать коллекцию в базе данных. Размерность векторов модели all-MiniLM-L6-v2 составляет 384, что указывается при создании коллекции. Для поиска схожих векторов используется косинусное расстояние.
collection_name = "documents"
vector_size = 384  # Размерность векторов модели all-MiniLM-L6-v2

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE) # Используем косинусное расстояние для поиска схожих векторов
)

In [ ]:
# 4. Подготавливаем текстовые данные (База знаний)
documents = [
    {"id": 1, "text": "Клетки являются базовыми строительными блоками всех живых организмов."},
    {"id": 2, "text": "Митохондрия — это энергетическая станция клетки, вырабатывающая АТФ."},
    {"id": 3, "text": "Опираться на векторный поиск удобно при построении RAG систем."},
    {"id": 4, "text": "Python — популярный язык программирования для анализа данных и машинного обучения."}
]

# 5. Генерируем векторы и сохраняем записи в Qdrant
points = []
for doc in documents:
    # Превращаем текст в вектор
    vector = embedder.encode(doc["text"]).tolist()
    
    # Собираем точечную структуру через готовый PointStruct (ID, Вектор, Исходные данные)
    points.append(
        PointStruct(
            id=doc["id"],
            vector=vector,
            payload={"text": doc["text"]}  # Метаданные
        )
    )

# Загружаем все точки в коллекцию
client.upsert(collection_name=collection_name, points=points)
print("Данные успешно индексированы в Qdrant!\n")

# 6. Выполняем семантический поиск
query_text = "Из чего состоит клетка человека?"
print(f"Поисковый запрос: '{query_text}'")

# Превращаем запрос пользователя в вектор
query_vector = embedder.encode(query_text).tolist()

# Выполняем поиск 2 самых похожих записей
search_results = client.query_points(
    collection_name=collection_name, # Название коллекции нужно указать, чтобы Qdrant знал, где искать
    query=query_vector,
    limit=2
)

# 7. Выводим результаты
print("\n--- Результаты поиска ---")
for hit in search_results.points:
    score = round(hit.score, 4)  # Косинусное сходство (чем ближе к 1.0, тем похоже)
    text = hit.payload["text"]
    print(f"[Score: {score}] {text}")

[PointStruct(id=1, vector=[0.011239193379878998, 0.0030982675962150097, -0.01644883118569851, 0.011624641716480255, -0.07378262281417847, 0.010784723795950413, 0.12546469271183014, 0.025538085028529167, -0.022302858531475067, 0.024834295734763145, -0.0157532449811697, 0.03624889627099037, 0.009192344732582569, 0.026526883244514465, -0.013254663906991482, 0.02339903451502323, -0.01833823136985302, 0.01047819759696722, -0.045864932239055634, 0.02858668565750122, -0.02193700708448887, -0.04880400747060776, 0.10670122504234314, 0.037344515323638916, -0.013180295936763287, 0.008592018857598305, 0.031465932726860046, -0.04664009064435959, 0.008858202956616879, 0.057488709688186646, -0.02134164422750473, -0.036591462790966034, 0.11581173539161682, -0.01318619679659605, -0.004375568591058254, -0.030244704335927963, -0.05167059600353241, -0.0050788805820047855, 0.0959806814789772, 0.061379168182611465, -0.05985565856099129, -0.061383411288261414, -0.06982065737247467, 0.036616772413253784, -0.0